# Landing-Page Skill Optimizer

This notebook accompanies Chapter 11 §11.1 of *Context Engineering with DSPy*: optimizing a landing-page generator skill end-to-end with `optimize_anything`.

The skill lives at `~/.claude/skills/landing-page/SKILL.md` and gets invoked when someone asks for a landing page. The input is a brief plus a brand guide. The output is a single-file `index.html` with inline CSS. We collect a handful of real failures by hand, define a rubric judge that returns per-dimension PASS/FAIL plus a composite score, and let GEPA's `optimize_anything` evolve the skill text against that metric.

## Required environment variables

Add this to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the page-generation and judge LM via `dspy.LM("openai/...")`.

## Required tooling

The rendered-HTML checks require Playwright's Chromium browser binary:

```bash
playwright install chromium
```

The notebook calls the candidate skill as a system prompt directly, so the Claude Code CLI is not required.

## Required assets

The dataset rows below reference `gold_html` and `brand_guide` artifacts you supply yourself — pages you've already shipped that you're happy with. Drop them under `./assets/landing-pages/` and update the file paths in the seed dataset. We don't ship sample HTML in the repo because the whole point of this loop is to optimize against *your* gold standard, not a generic one.


In [ ]:
%pip install -r ../requirements.txt -q
# optimize_anything is provided by GEPA 0.1.x, newer than the gepa==0.0.27
# that dspy 3.2.1 pins (and requirements.txt locks). This notebook was
# validated with 0.1.4; pip will report the intentional pin conflict.
%pip install -U "gepa==0.1.4" -q


## Setup

Load env vars and configure DSPy. The page-generation LM is `openai/gpt-5.6-sol` — strong enough that one rollout per example produces an honest page; we use a cheaper model for judging.


In [ ]:
import os, json, dspy
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not in env"

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))


## Step 1: turn failed sessions into a seed dataset

You don't need a hundred examples. You need five to ten real failures, captured by hand the moment a session goes wrong. Each row has the brief you handed the skill, the brand guide, what you wanted, and what specifically went wrong. We also point each row at a `gold_html` file — a page you shipped that you'd be happy if the skill matched.


In [ ]:
# A handful of real failures from the landing-page skill, captured by hand.
# Replace the gold_html paths with your own shipped pages.
seed_failures = [
    {
        "brief": "Acme Analytics. B2B analytics for ops teams. Single hero "
                 "section + email signup. Nothing else.",
        "brand_guide": {
            "colors":   {"primary": "#4338CA", "bg": "#FAFAFA", "text": "#111827"},
            "font":     "Inter",
            "voice":    "formal, technical, no hype words",
        },
        "expected": "One hero with headline, subhead, email field, submit. "
                    "Primary brand color on the CTA. No other sections.",
        "failure_mode": "scope_creep",
        "notes": "Agent shipped six sections (hero, features, pricing, "
                 "testimonials, FAQ, footer). Asked for one.",
        "gold_html": "./assets/landing-pages/acme_gold.html",
    },
    {
        "brief": "Outpost. Wilderness gear marketplace.",
        "brand_guide": {
            "colors":   {"primary": "#2D5016", "accent": "#C8A865", "bg": "#FFFCF5"},
            "font":     "Georgia",
            "voice":    "warm, conversational, human",
        },
        "expected": "Forest-green primary, serif typography, copy reads warm.",
        "failure_mode": "palette_drift",
        "notes": "Used #1F2937 (slate gray) instead of #2D5016. Copy was "
                 "'Revolutionize your gear shopping experience.' Generic.",
        "gold_html": "./assets/landing-pages/outpost_gold.html",
    },
    {
        "brief": "Cinder. Newsletter platform for indie writers. "
                 "Two sections: hero, three-feature grid.",
        "brand_guide": {
            "colors":   {"primary": "#DC2626", "bg": "#FFFFFF"},
            "font":     "system-ui",
            "voice":    "direct, opinionated, no marketing-speak",
        },
        "expected": "Hero + three-card feature row. Red CTA. No icons.",
        "failure_mode": "decoration_creep",
        "notes": "Added rocket / chart / lightbulb emoji icons to every "
                 "feature card. Brief said 'no icons.'",
        "gold_html": "./assets/landing-pages/cinder_gold.html",
    },
    # ... 4-7 more like this from your own usage.
]

dataset = seed_failures


## Step 2: programmatic checks

Five mechanical 0/1 checks. None of these need an LM, they all run in under a second, and together they form half of the composite score. Noise-free signal is worth more per unit than judge signal, which is why this block carries the largest weight in `grade_landing_page` below.


In [ ]:
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright


def check_no_horizontal_overflow(html: str, viewport_width: int = 375) -> int:
    """Render at the given viewport width and check for horizontal scroll."""
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch()
            page = browser.new_page(viewport={"width": viewport_width, "height": 800})
            page.set_content(html)
            scroll_w = page.evaluate("document.documentElement.scrollWidth")
            client_w = page.evaluate("document.documentElement.clientWidth")
            browser.close()
            return int(scroll_w <= client_w + 1)
    except Exception:
        return 0


def check_section_count_matches_brief(soup: BeautifulSoup, brief: str) -> int:
    """Crude heuristic: count <section> tags and check the brief didn't ask for a different number."""
    sections = len(soup.find_all("section"))
    brief_lower = brief.lower()
    if "single hero" in brief_lower or "one hero" in brief_lower:
        return int(sections <= 1)
    if "two sections" in brief_lower:
        return int(sections == 2)
    if "three sections" in brief_lower:
        return int(sections == 3)
    return 1  # no explicit budget asserted


def programmatic_checks(html: str, brand_guide: dict, brief: str) -> dict:
    """Five mechanical checks. No LM calls. All run in under a second."""
    soup = BeautifulSoup(html, "html.parser")

    # 1. Does it render at all?
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch()
            page = browser.new_page()
            page.set_content(html)
            browser.close()
            renders = 1
    except Exception:
        renders = 0

    # 2. Are the brand colors actually present in the CSS?
    css_text = " ".join(s.get_text() for s in soup.find_all("style"))
    style_attrs = " ".join(t.get("style", "") for t in soup.find_all(style=True))
    haystack = (css_text + style_attrs).lower()
    brand_colors_present = int(all(
        c.lower() in haystack for c in brand_guide["colors"].values()
    ))

    # 3. Is the brand font referenced?
    brand_font_used = int(brand_guide["font"].lower() in haystack)

    # 4. No horizontal overflow at mobile width (375px)?
    mobile_ok = check_no_horizontal_overflow(html, viewport_width=375)

    # 5. Did the agent stay within the section budget the brief asked for?
    section_budget_ok = check_section_count_matches_brief(soup, brief)

    return {
        "renders":              renders,
        "brand_colors_present": brand_colors_present,
        "brand_font_used":      brand_font_used,
        "mobile_no_overflow":   mobile_ok,
        "section_budget_ok":    section_budget_ok,
    }


## Step 3: pairwise judges

Two judges, both pairwise: one on a screenshot of the page (brand fit, visual craft), one on the visible text (copy voice). Pairwise is the right shape when you have a gold reference but can't articulate a rubric — "I'll know good when I see it." Each returns 0 or 1, position-randomized to neutralize the judge's first-response bias.


In [ ]:
class VisualBrandJudge(dspy.Signature):
    """Decide which screenshot is a better landing page for the given
    brief and brand guide. Judge against brand fit and visual craft, not
    which one is more 'designed-looking'."""
    brief:         str         = dspy.InputField()
    brand_guide:   dict        = dspy.InputField()
    screenshot_a:  dspy.Image  = dspy.InputField()
    screenshot_b:  dspy.Image  = dspy.InputField()
    reasoning:     str         = dspy.OutputField()
    winner:        str         = dspy.OutputField(desc="'A' or 'B'")


class CopyVoiceJudge(dspy.Signature):
    """Decide which page's copy better matches the brand voice."""
    brief:           str  = dspy.InputField()
    voice_rule:      str  = dspy.InputField()
    copy_a:          str  = dspy.InputField()
    copy_b:          str  = dspy.InputField()
    reasoning:       str  = dspy.OutputField()
    winner:          str  = dspy.OutputField(desc="'A' or 'B'")


## Step 4: wire judges into a pairwise helper

The `pairwise` helper randomizes which page is shown as A vs B, calls the judge, then maps the verdict back to "did the candidate win?". This is what kills first-response bias.


In [ ]:
import random


def _render_screenshot(html: str) -> dspy.Image:
    """Render HTML to a PNG screenshot wrapped as a dspy.Image."""
    import tempfile, base64
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(viewport={"width": 1280, "height": 800})
        page.set_content(html)
        png = page.screenshot(full_page=True)
        browser.close()
    b64 = base64.b64encode(png).decode("ascii")
    return dspy.Image(f"data:image/png;base64,{b64}")


def _visible_text(html: str) -> str:
    return BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)


def pairwise(judge_signature, gold_html: str, candidate_html: str, brand_guide: dict, brief: str = "") -> int:
    """Position-randomized pairwise judge. Returns 1 if candidate wins, else 0."""
    swap = random.random() < 0.5
    judge = dspy.Predict(judge_signature)

    if judge_signature is VisualBrandJudge:
        a_html, b_html = (candidate_html, gold_html) if not swap else (gold_html, candidate_html)
        verdict = judge(
            brief=brief,
            brand_guide=brand_guide,
            screenshot_a=_render_screenshot(a_html),
            screenshot_b=_render_screenshot(b_html),
        ).winner.strip().upper()
    else:  # CopyVoiceJudge
        a_html, b_html = (candidate_html, gold_html) if not swap else (gold_html, candidate_html)
        verdict = judge(
            brief=brief,
            voice_rule=brand_guide.get("voice", ""),
            copy_a=_visible_text(a_html),
            copy_b=_visible_text(b_html),
        ).winner.strip().upper()

    # If swap=False, candidate was A; candidate wins iff winner == 'A'.
    # If swap=True,  candidate was B; candidate wins iff winner == 'B'.
    candidate_won = (verdict == ("A" if not swap else "B"))
    return int(candidate_won)


## Step 5: the composite metric

50% mechanical, 25% visual judge, 25% copy judge. The weighting encodes a judgment call — noise-free signal is worth more per unit than judge signal. You can change it; the optimizer doesn't care.


In [ ]:
def grade_landing_page(html: str, example: dict) -> tuple[float, dict]:
    checks = programmatic_checks(html, example["brand_guide"], example["brief"])
    prog_score = sum(checks.values()) / len(checks)            # 0.0 - 1.0

    gold_html = open(example["gold_html"]).read()

    visual_win = pairwise(
        VisualBrandJudge, gold_html, html, example["brand_guide"], example["brief"],
    )                                                          # 0 or 1
    voice_win = pairwise(
        CopyVoiceJudge,   gold_html, html, example["brand_guide"], example["brief"],
    )                                                          # 0 or 1

    # Weighted composite. Mechanical checks are the largest slice because
    # they're noise-free; the two pairwise judges split the remainder.
    score = 0.5 * prog_score + 0.25 * visual_win + 0.25 * voice_win
    breakdown = {**checks, "visual_win": visual_win, "voice_win": voice_win}
    return score, breakdown


## Step 6: run the candidate skill against an example

`run_landing_page_skill` is what `optimize_anything` will call once per rollout: it takes a candidate SKILL.md, runs it as the system prompt for the page-generation LM, and returns the rendered HTML. (At production scale you'd swap in a `claude` subprocess that loads the candidate from `~/.claude/skills/landing-page/SKILL.md`; the direct-LM version keeps this notebook self-contained.)


In [ ]:
def run_landing_page_skill(skill_text: str, brief: str, brand_guide: dict) -> str:
    """Run the candidate SKILL.md as the system prompt; return rendered HTML."""
    page_lm = dspy.LM("openai/gpt-5.6-sol")
    return page_lm(messages=[
        {"role": "system", "content": skill_text},
        {"role": "user",   "content":
            f"Brief: {brief}\n\nBrand guide:\n{json.dumps(brand_guide, indent=2)}"},
    ])[0]


def evaluator(candidate: str, example: dict) -> tuple[float, dict]:
    # `candidate` is the current SKILL.md text. `example` is one row from dataset.
    html = run_landing_page_skill(candidate, example["brief"], example["brand_guide"])
    score, breakdown = grade_landing_page(html, example)
    side_info = {
        "brief":        example["brief"],
        "brand_guide":  example["brand_guide"],
        "generated":    html[:2000],   # first 2k chars is enough for reflection
        "expected":     example["expected"],
        "failure_mode": example.get("failure_mode"),
        "breakdown":    breakdown,     # per-dimension scores
    }
    return score, side_info


## Step 7: optimize the skill

`optimize_anything` introspects the evaluator's signature — the second kwarg **must** be named `example`. Rename it to `task` or `item` and GEPA silently drops the data, then you get a missing-arg crash several layers deep.

The cell below uses `max_metric_calls=3` so it finishes in under a minute and confirms the wiring. The next cell is the full-budget version (commented out) — uncomment once your env runs end-to-end. Each full rollout is one real page generation plus two judge calls — about $0.10 each — so the full run is on the order of a few dollars.


In [ ]:
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig

# Read the current SKILL.md from disk. Update this path to wherever your
# landing-page skill lives (e.g. ~/.claude/skills/landing-page/SKILL.md).
SEED_SKILL_PATH = os.path.expanduser("~/.claude/skills/landing-page/SKILL.md")
seed_candidate = open(SEED_SKILL_PATH).read()

result = optimize_anything(
    seed_candidate=seed_candidate,
    evaluator=evaluator,
    dataset=dataset,
    objective="Generate landing pages that match the brand guide and the brief.",
    config=GEPAConfig(engine=EngineConfig(max_metric_calls=3)),
)

print(result.best_candidate)


In [ ]:
# Full reproduction of the book's §11.1 run.
# Uncomment to run — takes ~30 minutes and a few dollars.
#
# result = optimize_anything(
#     seed_candidate=seed_candidate,
#     evaluator=evaluator,
#     dataset=dataset,
#     objective="Generate landing pages that match the brand guide and the brief.",
#     config=GEPAConfig(engine=EngineConfig(max_metric_calls=200)),
# )
#
# print(result.best_candidate)


Save the winning candidate over your existing `SKILL.md` (keep the previous version somewhere — regressions happen and the artifact is the only way to diagnose them) and the next session that loads the landing-page skill gets the improved version. As new failures accumulate, append them to `seed_failures` and rerun.
